# Format tables of NSQIP comparison results

In [ ]:
import pandas as pd
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import numpy as np

In [ ]:
NSQIP_RES_DIR = BASE_PATH / "results" / "nsqip_eval"
nsqip_res_df = pd.read_excel(NSQIP_RES_DIR / "raw_table.xlsx", index_col=0)

In [ ]:
def get_from_nsqip(outcome_name, model_name, metric_name, df, req_val):
    cond = (
        (nsqip_res_df["outcome_name"] == outcome_name)
        & (nsqip_res_df["model_name"] == model_name)
        & (nsqip_res_df["metric"] == metric_name)
    )
    return nsqip_res_df[cond][req_val].values[0]


ordered_cols = [
    "model",
    "AUROC (95% CI)",
    "AUROC p-val",
    "AUPRC (95% CI)",
    "AUPRC p-val",
    "brier",
    "Brier p-val",
    "Threshold",
    "f1",
    "accuracy",
    "recall",
    "precision",
    "ici",
]
us_res_table_dir = BASE_PATH / "results" / "class_report_tables"

In [ ]:
for outcome_dir in us_res_table_dir.iterdir():
    outcome_name = outcome_dir.name
    print(outcome_name)
    outcome_path = outcome_dir / "test.tsv"
    us_outcome_table = pd.read_csv(outcome_path, sep="\t", index_col=0)
    ## ADD p-vals
    us_outcome_table["AUROC p-val"] = ["NA" for i in range(len(us_outcome_table))]
    us_outcome_table["AUPRC p-val"] = ["NA" for i in range(len(us_outcome_table))]
    us_outcome_table["Brier p-val"] = ["NA" for i in range(len(us_outcome_table))]
    if outcome_name == "bleed":
        new_row = {key: "NA" for key in ordered_cols if key != "model"}
        new_row["model"] = "NSQIP"
        us_outcome_table.loc[len(us_outcome_table)] = new_row
    else:
        us_outcome_table["AUROC p-val"] = np.zeros(len(us_outcome_table)).astype(str)
        us_outcome_table["AUPRC p-val"] = np.zeros(len(us_outcome_table)).astype(str)
        us_outcome_table["Brier p-val"] = np.zeros(len(us_outcome_table)).astype(str)
        for model_name in us_outcome_table["model"]:
            row_cond = us_outcome_table["model"] == model_name
            ### ROC
            roc_p = get_from_nsqip(
                outcome_name=outcome_name,
                model_name=model_name,
                metric_name="auroc",
                df=nsqip_res_df,
                req_val="p",
            )
            us_outcome_table.loc[row_cond, "AUROC p-val"] = roc_p
            ### PRC
            prc_p = get_from_nsqip(
                outcome_name=outcome_name,
                model_name=model_name,
                metric_name="avg_prec",
                df=nsqip_res_df,
                req_val="p",
            )
            us_outcome_table.loc[row_cond, "AUPRC p-val"] = prc_p
            ### Brier
            brier_p = get_from_nsqip(
                outcome_name=outcome_name,
                model_name=model_name,
                metric_name="brier",
                df=nsqip_res_df,
                req_val="p",
            )
            us_outcome_table.loc[row_cond, "Brier p-val"] = brier_p
        ## Add new row
        nsqip_roc_val = get_from_nsqip(
            outcome_name=outcome_name,
            model_name=model_name,
            metric_name="auroc",
            df=nsqip_res_df,
            req_val="val (CIs)",
        )
        nsqip_prc_val = get_from_nsqip(
            outcome_name=outcome_name,
            model_name=model_name,
            metric_name="avg_prec",
            df=nsqip_res_df,
            req_val="val (CIs)",
        )
        nsqip_brier_val = get_from_nsqip(
            outcome_name=outcome_name,
            model_name=model_name,
            metric_name="brier",
            df=nsqip_res_df,
            req_val="val (CIs)",
        )
        new_row = {
            "model": "NSQIP",
            "AUROC (95% CI)": nsqip_roc_val,
            "AUROC p-val": "REFERENCE",
            "AUPRC (95% CI)": nsqip_prc_val,
            "AUPRC p-val": "REFERENCE",
            "brier": nsqip_brier_val,
            "Brier p-val": "REFERENCE",
            "Threshold": "NA",
            "f1": "NA",
            "accuracy": "NA",
            "recall": "NA",
            "precision": "NA",
            "ici": "NA",
        }
        us_outcome_table.loc[len(us_outcome_table)] = new_row
    ## Format + export
    outcome_table_ordered = us_outcome_table[ordered_cols]
    display(outcome_table_ordered)
    exp_path = NSQIP_RES_DIR / "report_tables" / f"{outcome_name}.csv"
    if exp_path.exists():
        exp_path.unlink()
    exp_path.parent.mkdir(exist_ok=True, parents=True)
    outcome_table_ordered.to_csv(exp_path)